In [42]:
import pyarrow.dataset as ds

In [ ]:
from pprint import pprint
from pathlib import Path
import yaml
from gelos.extraction import extract_embeddings
from gelos.transforms import TRANSFORMS
from gelos.models import MODELS
from gelos.plotting import PLOTS, build_style_from_config

In [68]:
yaml_path = Path("/app/configs/exp001_prithvi300.yaml")
config_stem = yaml_path.stem
raw_data_dir = Path("/app/data/raw")
embedding_dir = Path("/app/data/interim")
with open(yaml_path, "r") as f:
    yaml_config = yaml.safe_load(f)
version = yaml_config['data_version']
figures_dir = Path("/app/reports/figures")
output_dir = embedding_dir / version / config_stem

In [37]:
embedding_dirs = list(output_dir.glob("*"))
for dir in embedding_dirs:
    print(str(dir))

/app/data/interim/v0.50.1/exp001_prithvi300/layer_23


In [72]:
embedding_layer = "layer_23"
embeddings_directory = output_dir / embedding_layer

In [64]:
print("Available straegies in config:\n------------")
for strategy_key, strategy_cfg in yaml_config["embedding_extraction_strategies"].items():
    print(f"{strategy_cfg['title']} -  strategy key: {strategy_key}")

Available straegies in config:
------------
CLS Token -  strategy key: cls_token
All Steps of Middle Patch -  strategy key: all_steps_of_middle_patch
All Patches from April to June -  strategy key: all_patches_from_april_to_june


In [65]:
strategy_key = "all_steps_of_middle_patch"

In [69]:
slice_args = strategy_cfg["slice_args"]
strategy_title = strategy_cfg.get("title", strategy_key)
prefix = f"{config_stem}_{strategy_key}_{embedding_layer}"

In [71]:
layer_dir = output_dir / embedding_layer
emb_cache = layer_dir / f"{prefix}_embeddings.npy"
idx_cache = layer_dir / f"{prefix}_chip_indices.npy"

if emb_cache.exists() and idx_cache.exists():
    print(f"loading cached embeddings from {emb_cache}")
    embeddings = np.load(emb_cache)
    chip_indices = np.load(idx_cache).tolist()
else:
    print(
        f"extracting embeddings: layer={embedding_layer}, "
        f"strategy={strategy_key}"
    )
    embeddings, chip_indices = extract_embeddings(
        embeddings_directory, slice_args=slice_args
    )
    layer_dir.mkdir(exist_ok=True, parents=True)
    np.save(emb_cache, embeddings)
    np.save(idx_cache, np.array(chip_indices))
    print(f"cached embeddings to {emb_cache}")

extracting embeddings: layer=layer_23, strategy=all_steps_of_middle_patch


Processing embeddings: 45720it [02:26, 315.90it/s]

In [ ]:
# Step 2: transform
tsne_result = TRANSFORMS["tsne"](embeddings, perplexity=30)

In [ ]:
# Step 3: plot or model
MODELS["knn"](embeddings, labels, output_dir=Path("/app/reports"), run_name="test")